[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/agents/langgraph/langgraph_parallel.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>

# LangGraph：并行评估

LangGraph 并行评估
本教程使用 LangGraph 构建并行执行工作流，适用于多个评估或子任务可以独立运行，随后汇总为最终决策的场景。

应用先生成有吸引力的产品描述，再并行运行三项检查：

- 安全检查：内容是否安全且不含暴力？

- 政策合规：是否符合公司政策？

- 清晰度检查：普通读者是否能理解？

这一模式展示如何在共同的生成步骤之后分散执行，并在最终输出前汇总结果。

使用 Phoenix 追踪可以完整查看每个节点的执行，便于调试或审查各并行分支的决策过程。

In [ ]:
!pip install langgraph langchain langchain_community "arize-phoenix" arize-phoenix-otel openinference-instrumentation-langchain langchain-openai

In [ ]:
import os
from getpass import getpass

from langgraph.graph import END, START, StateGraph

In [ ]:
os.environ["OPENAI_API_KEY"] = getpass("🔑 请输入 OpenAI API 密钥： ")

# 配置 Phoenix 追踪
请前往 https://app.phoenix.arize.com/ 生成 API 密钥，以便使用 Phoenix 追踪 LangGraph 应用。

In [ ]:
if "PHOENIX_API_KEY" not in os.environ:
    os.environ["PHOENIX_API_KEY"] = getpass("🔑 请输入 Phoenix API 密钥： ")

if "PHOENIX_COLLECTOR_ENDPOINT" not in os.environ:
    os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = getpass("🔑 请输入 Phoenix 收集器端点： ")

In [ ]:
from phoenix.otel import register

tracer_provider = register(project_name="Parallel", auto_instrument=True)

In [ ]:
from langchain_openai import ChatOpenAI
from typing_extensions import TypedDict

# 选择 LLM

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

# 定义图状态
定义 State 对象以跟踪 LangGraph 中流动的所有数据，包括输入的产品名称、生成的描述、三项独立评估结果和最终汇总输出。

In [ ]:
class State(TypedDict):
    product: str
    description: str
    safety_check: str
    policy_check: str
    clarity_check: str
    final_output: str

# 节点 1：生成产品描述
此节点使用 LLM 撰写有吸引力的营销风格产品描述，并将输出存入图状态的 description 字段。

In [ ]:
def generate_description(state: State):
    msg = llm.invoke(f"为以下产品撰写有吸引力的产品描述： {state['product']}")
    return {"description": msg.content}

# 节点 2–4：并行评估检查
生成产品描述后，执行分为三个评估器，各自并行完成独立检查：

**安全检查**：语言是否安全且不含暴力？

**政策合规**：是否符合公司规范？

**清晰度检查**：普通读者是否能理解？

每个函数都接收同一份描述，并返回二元决策（"yes" 或 "no"）。

In [ ]:
company_policy = """公司产品描述政策
1. 语气与语言
产品描述必须：

使用清晰、简洁、专业的语言。

保持友好、乐于助人、包容的语气。

避免俚语、粗话、讽刺或过度随意的表达。

不含冒犯、歧视或文化不敏感的词语。

2. 真实性与准确性
所有产品功能、规格与收益必须符合事实。

‘最快’‘同类最佳’等主张必须可以验证，或有奖项、基准测试等证据支持。

避免：

误导性夸大。

无依据的健康或性能主张。

除非有法律依据，否则不使用‘保证’或‘无风险’。

3. 法律与监管合规
描述不得：

承诺‘能够治愈’‘保证成功’等结果，除非符合 FDA/FTC 要求。

在受监管行业中未经认证使用‘有机’‘非转基因’‘医用级’等受限术语。

使用贬损或不客观的品牌比较语言。

4. 品牌声音与一致性
产品语言必须符合品牌价值观：

赋能

可持续性

创新

信任

适用时始终提及独特卖点（USP），包括体现公司使命的差异化特征。

5. 包容性与可访问性
避免带有性别偏见、年龄歧视或文化排他性的表达。

使用具有十年级及以下阅读水平的人也能理解的平实语言。

除非必要，避免小众引用、习语或地域特有表达。

6. 格式与风格
使用正确的语法、标点与拼写。

标题与要点应当：

格式一致。

列举功能时以动词开头，例如‘提升清晰度’‘自动同步’。

7. 限制内容
产品描述不得包含：

暴力、武器或军事化比较。

露骨或暗示性的性内容。

非法活动或物质。

政治或宗教背书。

8. SEO 与可发现性（可选）
若进行搜索优化：

使用批准的关键词，密度保持低于 2.5%。

不要生硬堆砌关键词或影响可读性。

根据上下文自然加入相关标签。

9. 行动号召（CTA）准则
行动号召应当：

清楚并指向行动，例如‘立即购买’‘体验不同’。

避免强推措辞，例如‘再不行动就没了！’。

语气自信，但不咄咄逼人。

"""

In [ ]:
def check_safety(state: State):
    msg = llm.invoke(
        f"此产品描述是否安全且不含暴力？回答 yes 或 no。\n\n{state['description']}"
    )
    return {"safety_check": msg.content}


def check_policy(state: State):
    msg = llm.invoke(
        f"此产品描述是否符合公司政策？公司政策如下： {company_policy}。回答 yes 或 no。\n\n{state['description']}"
    )
    return {"policy_check": msg.content}


def check_clarity(state: State):
    msg = llm.invoke(
        f"十年级读者是否能清楚理解此描述？回答 yes 或 no。\n\n{state['description']}"
    )
    return {"clarity_check": msg.content}

# 节点 5：汇总结果
所有检查完成后，此节点收集回复。全部返回 "yes" 时通过产品描述，否则标记为拒绝，并附上原因。

In [ ]:
def aggregate_results(state: State):
    if (
        "yes" in state["safety_check"].strip().lower()
        and "yes" in state["policy_check"].strip().lower()
        and "yes" in state["clarity_check"].strip().lower()
    ):
        return {"final_output": state["description"]}
    return {
        "final_output": "已拒绝：一项或多项检查失败。\n"
        f"安全： {state['safety_check']}, 政策： {state['policy_check']}, 清晰度： {state['clarity_check']}"
    }

# 构建并行评估图
定义全部节点后，使用 StateGraph 将其组装为 LangGraph。

**开始 → 描述**：首先生成产品描述。

**分散检查**：输出进入安全、政策、清晰度三个并行分支，同时执行验证。

**汇合 → 汇总**：全部检查完成后，结果汇入最终汇总节点，决定通过或拒绝描述。

**结束**：生成最终结果。

此配置展示了 LangGraph 管理并行与汇合的能力，在保持透明和模块化的同时简化复杂工作流。

In [ ]:
builder = StateGraph(State)

builder.add_node("generate_description", generate_description)
builder.add_node("check_safety", check_safety)
builder.add_node("check_policy", check_policy)
builder.add_node("check_clarity", check_clarity)
builder.add_node("aggregate_results", aggregate_results)

# 首先生成描述
builder.add_edge(START, "generate_description")

# 随后分支执行并行检查
builder.add_edge("generate_description", "check_safety")
builder.add_edge("generate_description", "check_policy")
builder.add_edge("generate_description", "check_clarity")

# 所有检查均流向汇总器
builder.add_edge("check_safety", "aggregate_results")
builder.add_edge("check_policy", "aggregate_results")
builder.add_edge("check_clarity", "aggregate_results")

# 最终结果
builder.add_edge("aggregate_results", END)

workflow = builder.compile()

# 使用示例

In [ ]:
state = workflow.invoke({"product": "Smart glasses that project your calendar"})
print(state["final_output"])

In [ ]:
state = workflow.invoke(
    {"product": "Headphones with noise cancellation, transparency, and other advanced features."}
)
print(state["final_output"])
state = workflow.invoke({"product": "Smart fridge with advanced features."})
print(state["final_output"])

# 记得在 Phoenix 中查看追踪！

# 添加评估（Evals）

本节通过另一次 LLM 调用，评估安全、政策和清晰度检查器的准确性。

In [ ]:
from phoenix.client import Client
from phoenix.client.types.spans import SpanQuery

df = Client().spans.get_spans_dataframe(
    query=SpanQuery().where("name == 'LangGraph'"), project_name="Parallel"
)

In [ ]:
df.to_csv("parallel_evals.csv")

# 自定义评估模板

这里定义自定义评估模板，用于评估政策、清晰度和安全检查器的决策。

In [ ]:
TEMPLATE = """ 你必须判断清晰度、政策和安全检查器是否根据生成的描述作出正确决策。
检查政策检查器的决策是否正确反映生成描述对以下政策的遵守情况：{company_policy}。
检查清晰度检查器的决策是否正确反映十年级读者能否理解生成描述。
检查安全检查器的决策是否正确反映生成描述是否安全且不含暴力。

生成的描述：{description}
决策：{decisions}

一个决策正确时输出 1/3，两个正确输出 2/3，全部正确输出 3/3，全部错误输出 0/3。
解释推理过程。
"""

In [ ]:
df

# 生成评估

In [ ]:
import json

import pandas as pd
from phoenix.evals import LLM, ClassificationEvaluator, async_evaluate_dataframe


def unpack(row):
    blob = json.loads(row["attributes.output.value"])
    # 提取自由文本描述
    description = blob.get("description", "")
    # 将三个 yes/no 标记合并为可读字符串
    decisions_dict = {
        "policy_checker": blob.get("policy_check", "No."),
        "clarity_checker": blob.get("clarity_check", "No."),
        "safety_checker": blob.get("safety_check", "No."),
    }
    # 拼接为如下形式："policy_checker: Yes., clarity_checker: No.,  ..."
    decisions = ", ".join(f"{k}: {v}" for k, v in decisions_dict.items())
    return pd.Series({"description": description, "decisions": decisions})


df[["description", "decisions"]] = df.apply(unpack, axis=1)

# 3. 确保 TEMPLATE 的每个 {placeholder} 都有对应列 --------
df["company_policy"] = company_policy  # 现在可以解析 {company_policy}

# 4. 运行评估 --------------------------------------------------------------

# 将 0/3 至 3/3 视为四个类别
rails = ["0/3", "1/3", "2/3", "3/3"]

checker_evaluator = ClassificationEvaluator(
    name="checker_accuracy",
    llm=LLM(provider="openai", model="gpt-4o"),
    prompt_template=TEMPLATE,
    choices=rails,
)

eval_results = await async_evaluate_dataframe(
    dataframe=df,
    evaluators=[checker_evaluator],
    concurrency=10,
)

In [ ]:
eval_results[["checker_accuracy_score", "checker_accuracy_execution_details"]]

# 将评估导出到 Phoenix！

In [ ]:
from phoenix.client import AsyncClient
from phoenix.evals.utils import to_annotation_dataframe

px_client = AsyncClient()

annotation_df = to_annotation_dataframe(eval_results, ["checker_accuracy"])
await px_client.spans.log_span_annotations_dataframe(
    dataframe=annotation_df,
    annotator_kind="LLM",
)